# BEIP — EDA 02: Candidate Affidavits Analysis (MyNeta / ADR)

**Objective:** Explore candidate affidavits from `silver.candidate_affidavits` joined with electoral performance to analyze:
- Affidavit coverage across election years
- Influence of criminal cases on election outcomes
- Wealth & asset distributions
- Educational attainment and its correlation with victory
- Correlations between candidate profile attributes and winning

## 1. Setup and Data Loading

> **Note:** The affidavits table does not have `state_name` (it was not in the scraped source data),
> so we join on `year + constituency_name + candidate_name` only.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

sys.path.insert(0, str(Path.cwd().parent))
from src.config import get_engine

engine = get_engine()

# Join on year + constituency + candidate (state_name is NULL in scraped data)
query = """
    SELECT
        a.year,
        e.state_name,
        a.constituency_name,
        a.candidate_name,
        a.party,
        a.criminal_cases,
        a.serious_criminal_cases,
        a.has_criminal_case,
        a.has_serious_case,
        a.education,
        a.total_assets,
        a.total_liabilities,
        a.is_crorepati,
        e.position,
        e.votes,
        e.vote_share,
        CASE WHEN e.position = 1 THEN 1 ELSE 0 END AS won
    FROM silver.candidate_affidavits a
    LEFT JOIN silver.election_results e
        ON  a.year              = e.year
        AND a.constituency_name = e.constituency_name
        AND a.candidate_name    = e.candidate_name
"""

df = pd.read_sql(query, engine)

# Force numeric types upfront
df["total_assets"]           = pd.to_numeric(df["total_assets"],           errors="coerce")
df["total_liabilities"]      = pd.to_numeric(df["total_liabilities"],      errors="coerce")
df["criminal_cases"]         = pd.to_numeric(df["criminal_cases"],         errors="coerce").fillna(0).astype(int)
df["serious_criminal_cases"] = pd.to_numeric(df["serious_criminal_cases"], errors="coerce").fillna(0).astype(int)

# Re-derive flags from clean numeric data
df["has_criminal_case"] = df["criminal_cases"] > 0
df["has_serious_case"]  = df["serious_criminal_cases"] > 0
df["is_crorepati"]      = df["total_assets"] > 1e7

matched = df["position"].notna().sum()
print(f"Total affidavit records : {len(df):,}")
print(f"Matched to elections    : {matched:,} ({matched/len(df)*100:.1f}%)")
print(f"Rows with valid assets  : {df['total_assets'].notna().sum():,}")

## 2. Data Overview & Quality Check

In [ ]:
df.head()

In [ ]:
print("=== Missing / Null Summary ===")
print(df.isnull().sum())

print("\n=== Candidate Count by Year ===")
display(df["year"].value_counts().sort_index())

## 3. Criminal Cases Analysis

In [ ]:
total        = len(df)
with_cases   = df["has_criminal_case"].sum()
with_serious = df["has_serious_case"].sum()

print(f"Total Candidates:               {total:,}")
print(f"With >= 1 Criminal Case:        {with_cases:,}  ({with_cases/total*100:.1f}%)")
print(f"With Serious Criminal Cases:    {with_serious:,}  ({with_serious/total*100:.1f}%)")

party_cases = df.groupby("party").agg(
    total=("candidate_name", "count"),
    pct_with_cases=("has_criminal_case", lambda x: x.mean() * 100)
).reset_index()

top_party_cases = (
    party_cases[party_cases["total"] >= 30]
    .sort_values("pct_with_cases", ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_party_cases, x="pct_with_cases", y="party", palette="Reds_r")
plt.title("% Candidates with Criminal Cases by Party (Min 30)", fontsize=14, fontweight="bold")
plt.xlabel("% Candidates with Cases")
plt.ylabel("Party")
plt.show()

In [ ]:
# Only include rows where we have electoral outcome
df_matched = df[df["position"].notna()].copy()

crime_win = df_matched.groupby("has_criminal_case").agg(
    count=("candidate_name", "count"),
    win_rate_pct=("won", lambda x: x.mean() * 100),
    avg_vote_share=("vote_share", "mean")
).reset_index()

crime_win["has_criminal_case"] = crime_win["has_criminal_case"].map({True: "Has Cases", False: "Clean Record"})
display(crime_win)

plt.figure(figsize=(6, 5))
sns.barplot(data=crime_win, x="has_criminal_case", y="win_rate_pct", palette="Set2")
plt.title("Win Rate: Criminal Record vs. Clean", fontsize=13, fontweight="bold")
plt.ylabel("Win Rate (%)")
plt.xlabel("")
plt.show()

## 4. Candidate Wealth & Assets

In [ ]:
print("=== Declared Assets Summary (₹ Crores) ===")
print((df["total_assets"] / 1e7).describe())

valid_assets = df["total_assets"].dropna()
if len(valid_assets) > 0 and valid_assets.max() > 0:
    log_assets = np.log10(valid_assets.clip(lower=1))
    plt.figure(figsize=(12, 5))
    sns.histplot(x=log_assets, bins=40, kde=True, color="darkgreen")
    plt.title("Log10 Distribution of Declared Assets (INR)", fontsize=14, fontweight="bold")
    plt.xlabel("Log10(Total Assets in INR)")
    plt.ylabel("Candidate Count")
    plt.show()
else:
    print("[INFO] Asset data not in numeric format — likely stored as text in the Bronze table.")

In [ ]:
wealth_win = df_matched.groupby("is_crorepati").agg(
    count=("candidate_name", "count"),
    win_rate_pct=("won", lambda x: x.mean() * 100),
    avg_vote_share=("vote_share", "mean")
).reset_index()

wealth_win["is_crorepati"] = wealth_win["is_crorepati"].map({True: "Crorepati (>₹1Cr)", False: "Non-Crorepati"})
display(wealth_win)

if len(wealth_win) > 1:
    plt.figure(figsize=(6, 5))
    sns.barplot(data=wealth_win, x="is_crorepati", y="win_rate_pct", palette="Blues_d")
    plt.title("Win Rate by Wealth Category", fontsize=13, fontweight="bold")
    plt.ylabel("Win Rate (%)")
    plt.xlabel("")
    plt.show()
else:
    print("[INFO] Only one wealth category — asset data may all be null.")

## 5. Education Level Analysis

In [ ]:
edu_stats = df_matched.groupby("education").agg(
    candidate_count=("candidate_name", "count"),
    win_rate_pct=("won", lambda x: x.mean() * 100)
).reset_index().sort_values("win_rate_pct", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=edu_stats, x="win_rate_pct", y="education", palette="mako")
plt.title("Win Rate (%) by Declared Education Level", fontsize=14, fontweight="bold")
plt.xlabel("Win Rate (%)")
plt.ylabel("Education Level")
plt.show()

display(edu_stats)

## 6. Correlation Heatmap

In [ ]:
numeric_cols = ["criminal_cases", "serious_criminal_cases", "total_assets", "total_liabilities", "vote_share", "won"]
available    = [c for c in numeric_cols if c in df_matched.columns]

corr = df_matched[available].dropna().corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix: Candidate Attributes vs. Outcomes", fontsize=14, fontweight="bold")
plt.show()